<a href="https://colab.research.google.com/github/shivang5209/phishing-website-detector/blob/master/phishing_detector_phiusiil_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))



Saving PhiUSIIL_Phishing_URL_Dataset_engineered.csv to PhiUSIIL_Phishing_URL_Dataset_engineered.csv
['PhiUSIIL_Phishing_URL_Dataset_engineered.csv']


In [2]:
import pandas as pd

df = pd.read_csv("/content/PhiUSIIL_Phishing_URL_Dataset_engineered.csv")
print("shape:", df.shape)
print("columns:", df.columns.tolist())
print(df.head(2))


shape: (235795, 55)
columns: ['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']
                                URL  URLLength                  

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

# 1) Load data
df = pd.read_csv("/content/PhiUSIIL_Phishing_URL_Dataset_engineered.csv")

# 2) Auto-detect label column
label_candidates = ["label", "Label", "class", "Class", "Result", "result", "target", "Target", "is_phishing"]
label_col = next((c for c in label_candidates if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Available columns: {df.columns.tolist()}")

# 3) Build X, y
X = df.drop(columns=[label_col])
y = df[label_col]

# Keep only numeric columns to avoid string issues
X = X.select_dtypes(include=["number"]).fillna(0)

# 4) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5) Train model
model = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

# 6) Evaluate
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
print("Accuracy:", round(acc, 4))
print(classification_report(y_test, pred))

# 7) Save artifact
joblib.dump(model, "/content/model_artifact_phiusiil.pkl")
print("Saved: /content/model_artifact_phiusiil.pkl")


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20189
           1       1.00      1.00      1.00     26970

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159

Saved: /content/model_artifact_phiusiil.pkl


In [4]:
import pandas as pd

df = pd.read_csv("/content/PhiUSIIL_Phishing_URL_Dataset_engineered.csv")
sus = [c for c in df.columns if any(k in c.lower() for k in ["label","class","target","phish","result"])]
print("Suspicious column names:", sus)

imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
print("\nTop 20 features:\n", imp)


Suspicious column names: ['label']

Top 20 features:
 URLSimilarityIndex            0.197422
NoOfExternalRef               0.165641
LineOfCode                    0.131394
NoOfImage                     0.112452
NoOfSelfRef                   0.086916
NoOfJS                        0.075427
HasSocialNet                  0.032871
NoOfCSS                       0.030886
NoOfOtherSpecialCharsInURL    0.023214
HasDescription                0.023070
HasCopyrightInfo              0.022458
IsHTTPS                       0.015765
LargestLineLength             0.013440
NoOfDegitsInURL               0.009837
URLLength                     0.007053
NoOfLettersInURL              0.006501
DegitRatioInURL               0.006244
NoOfEmptyRef                  0.005181
HasSubmitButton               0.005093
DomainTitleMatchScore         0.004963
dtype: float64


In [5]:
import joblib

bundle = {
    "model": model,
    "feature_columns": X.columns.tolist(),
    "label_column": "label"
}
joblib.dump(bundle, "/content/model_bundle_phiusiil.pkl")
print("Saved:", "/content/model_bundle_phiusiil.pkl")
print("Feature count:", len(bundle["feature_columns"]))


Saved: /content/model_bundle_phiusiil.pkl
Feature count: 50


In [6]:
from google.colab import files
files.download("/content/model_bundle_phiusiil.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>